## Extraction (Api Rest & Web Scraping)

In [ ]:


import re
import time
import unicodedata
from pathlib import Path
from typing import Dict, Any

import pandas as pd
import requests
from bs4 import BeautifulSoup

# ================= CONFIG =================

SEASONS = [2021, 2022, 2023, 2024, 2025]
ERGAST = "https://api.jolpi.ca/ergast/f1"

# Racine du projet (on remonte depuis scripts/)
PROJECT_DIR = Path.cwd().parent

# Dossier data/raw
DATA_DIR = PROJECT_DIR / "data" / "raw"
DATA_DIR.mkdir(parents=True, exist_ok=True)


COST_CAP_BASELINE_USD = {
    2021: 145_000_000,
    2022: 140_000_000,
    2023: 135_000_000,
    2024: 135_000_000,
    2025: 135_000_000,
}

SALARY_SOURCES = {
    2021: "https://racingnews365.com/formula-1-driver-salaries-2021",
    2022: "https://racingnews365.com/what-are-the-driver-salaries-for-2022",
    2023: "https://racingnews365.com/f1-driver-salaries-2023",
    2024: "https://www.nbcnewyork.com/news/sports/how-much-are-f1-drivers-paid-salaries-2024/5192874/",
    2025: "https://racingnews365.com/2025-f1-driver-salaries-how-much-do-f1-drivers-earn",
}

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

# ================= UTILITIES =================

def get_json(url: str) -> Dict[str, Any]:
    r = requests.get(url, timeout=30, headers=HEADERS)
    r.raise_for_status()
    return r.json()

def normalize_text(s: str) -> str:
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]", "", s.lower())

def salary_to_usd(value: str):
    if not isinstance(value, str):
        return None
    value = value.lower().replace("$", "").replace(",", "")
    value = value.replace("million", "").replace("m", "").strip()

    match = re.search(r"(\d+(\.\d+)?)", value)
    if match:
        return int(float(match.group(1)) * 1_000_000)
    return None

# ================= ERGAST =================

def get_constructor_standings(season: int) -> pd.DataFrame:
    url = f"{ERGAST}/{season}/constructorStandings.json"
    data = get_json(url)
    standings = data["MRData"]["StandingsTable"]["StandingsLists"][0]["ConstructorStandings"]

    rows = []
    for s in standings:
        rows.append({
            "season": season,
            "constructorId": s["Constructor"]["constructorId"],
            "constructorName": s["Constructor"]["name"],
            "constructorRank": int(s["position"]),
            "constructorPoints": float(s["points"]),
            "costCapUSD": COST_CAP_BASELINE_USD[season]
        })
    return pd.DataFrame(rows)

def get_driver_standings(season: int) -> pd.DataFrame:
    url = f"{ERGAST}/{season}/driverStandings.json"
    data = get_json(url)
    standings = data["MRData"]["StandingsTable"]["StandingsLists"][0]["DriverStandings"]

    rows = []
    for s in standings:
        d = s["Driver"]
        constructor = s["Constructors"][0]
        rows.append({
            "season": season,
            "driverId": d["driverId"],
            "driverName": f'{d["givenName"]} {d["familyName"]}',
            "constructorId": constructor["constructorId"],
            "driverRank": int(s["position"]),
            "driverPoints": float(s["points"])
        })
    return pd.DataFrame(rows)

# ================= SALARY SCRAPING =================

def extract_salary_table(season: int, url: str) -> pd.DataFrame:
    html = requests.get(url, headers=HEADERS, timeout=30).text
    soup = BeautifulSoup(html, "html.parser")
    tables = soup.find_all("table")

    best_table = None
    for table in tables:
        headers = " ".join([th.get_text(strip=True).lower() for th in table.find_all("th")])
        if "driver" in headers and ("salary" in headers or "earn" in headers):
            best_table = table
            break

    if best_table is None:
        return pd.DataFrame()

    rows = []
    for tr in best_table.find_all("tr"):
        cells = [td.get_text(strip=True) for td in tr.find_all(["td", "th"])]
        if len(cells) >= 2:
            rows.append(cells)

    df = pd.DataFrame(rows[1:], columns=rows[0])
    df["season"] = season
    df["salarySourceURL"] = url
    return df

# ================= BUILD CSV =================

def build_constructor_finance():
    all_df = []
    for season in SEASONS:
        df = get_constructor_standings(season)
        all_df.append(df)
        print(f"✔ Constructors {season}")
        time.sleep(0.5)

    final = pd.concat(all_df, ignore_index=True)
    final.to_csv(DATA_DIR / "f1_constructor_finance_2021_2025.csv", index=False)
    print("✅ Constructor finance CSV créé")

def build_driver_finance():
    all_df = []

    for season in SEASONS:
        drivers = get_driver_standings(season)
        salary_table = extract_salary_table(season, SALARY_SOURCES[season])

        if not salary_table.empty:
            drivers["driverNorm"] = drivers["driverName"].apply(normalize_text)
            salary_table["driverNorm"] = salary_table.iloc[:,0].apply(normalize_text)

            merged = drivers.merge(
                salary_table[[salary_table.columns[0], salary_table.columns[1], "driverNorm", "salarySourceURL"]],
                on="driverNorm",
                how="left"
            )

            merged.rename(columns={
                salary_table.columns[1]: "salary_raw"
            }, inplace=True)

            merged["estimatedSalaryUSD"] = merged["salary_raw"].apply(salary_to_usd)

        else:
            merged = drivers
            merged["salary_raw"] = None
            merged["estimatedSalaryUSD"] = None
            merged["salarySourceURL"] = None

        all_df.append(merged)
        print(f"✔ Drivers {season}")
        time.sleep(0.5)

    final = pd.concat(all_df, ignore_index=True)

    final = final[[
        "season",
        "driverId",
        "driverName",
        "constructorId",
        "driverRank",
        "driverPoints",
        "salary_raw",
        "estimatedSalaryUSD",
        "salarySourceURL"
    ]]

    final.to_csv(DATA_DIR / "f1_driver_finance_2021_2025.csv", index=False)
    print("✅ Driver finance CSV créé")

# ================= MAIN =================

if __name__ == "__main__":
    build_constructor_finance()
    build_driver_finance()
    print("🎉 Extraction Finance terminée")
